# 12 - מרכזיות ברשת הרכבות ופרופיל סוציו-אקונומי

גרף הרכבות שנבנה במחברת 11 מלמד אותנו *היכן* רשת רכבת ישראל שברירית מבחינה מבנית. מחברת זו שואלת
את שאלת ההמשך: **האם שבריריות זו, ונפח השירות הנלווה אליה, מתיישבים עם הפרופיל הסוציו-אקונומי של
השכונות שבהן ממוקמות התחנות?** אנו משייכים כל תחנת רכבת לשכבת האזורים הסטטיסטיים הסוציו-אקונומיים
של הלמ"ס (CBS) לשנת 2021, משווים משפחות של מדדי מרכזיות בין קבוצות סוציו-אקונומיות ובין מחוזות
מנהליים, מסווגים כל תחנה לארכיטיפ מבני, ובוחנים את הקשרים הסוציו-אקונומיים באופן פורמלי.

**שאלת המחקר.** האם תחנות רכבת בעלות מרכזיות גבוהה, נפח שירות גבוה, או קריטיות מבנית, ממוקמות
באופן שיטתי באזורים סטטיסטיים עשירים יותר או עניים יותר?

**קלט**
- `outputs/nb/11_*/tables/rail_station_metrics.csv` - מרכזיות ברמת התחנה ברשת הרכבות (67 תחנות).
- `outputs/nb/11_*/tables/single_station_damage.csv` - תחנות המנותקות עקב תקלה בודדת ממודלת.
- `stops_with_socioeconomic.csv` - ה-join המרחבי של הפרויקט בין תחנות לבין אזורים סטטיסטיים של הלמ"ס
  (`socio_cluster`, `socio_index_value`, `region`, `metro`, שיטת ההתאמה והמרחק).
  שני הקלטים מאותרים אוטומטית; ראו את הסעיף "איתור הקלטים".

**פלט** (הכל תחת `outputs/nb/12_rail_socioeconomic/`)
- `tables/rail_station_archetypes.csv` - הטבלה המאוחדת המלאה, הכוללת אחוזונים, שני ציוני ה-composite
  ותווית הארכיטיפ.
- `tables/rail_centrality_spearman.csv` - מטריצת יתירות מדד מול מדד.
- `tables/composite_score_comparison.csv` - דירוג composite מקורי (legacy) מול דירוג מאוזן-משפחות.
- `tables/rail_socioeconomic_correlations.csv` - 24 מבחני Spearman **כולל** תיקון לריבוי השוואות.
- `tables/rail_socioeconomic_group_summary.csv`, `tables/rail_regional_summary.csv`.
- `figures/centrality_profile_heatmap.png`, `figures/service_vs_structural_damage.png`,
  `figures/socioeconomic_centrality_profiles.png`, `figures/regional_rail_profile.png`.

**ארבעה דברים שמחברת זו מתקנת או חושפת במפורש ביחס לגרסת הסקריפט הקודמת**
1. *סקאלות אחוזונים בלתי-ניתנות להשוואה במפת החום של הפרופיל.* האיור הישן הציב חמישה אחוזוני מרכזיות
   (המדורגים על פני כל 67 התחנות) לצד אחוזון נזק שדורג רק בקרב ~14 התחנות שהסרתן אכן מפרקת את הגרף,
   בעוד שסרגל הצבעים הצהיר "אחוזון מבין 67 תחנות רכבת". כאן עמודת הנזק מצוירת על **ציר נפרד משלה עם
   סרגל צבעים משלה ביחידות גולמיות**, ושתי ההגדרות של אחוזון הנזק מיוצאות זו לצד זו תחת שמות מפורשים.
2. *ציון composite יתיר.* ממוצע של חמישה אחוזונים סופר את נפח השירות שלוש פעמים
   (`weighted_degree` מול `scheduled_stop_calls`, Spearman rho = 0.94; `weighted_degree` מול
   `pagerank` = 0.86). אנו שומרים את הציון המקורי לצורך המשכיות, אך הופכים את ה-composite
   **המאוזן-משפחות** למספר המוביל ומראים כיצד שני הדירוגים נבדלים.
3. *24 מבחני Spearman ללא תיקון.* אנו מוסיפים תיקוני Benjamini-Hochberg ו-Holm ומדווחים את התוצאה
   בכנות: **דבר אינו מובהק, אף לא לפני התיקון.** תוצאה שלילית זו היא הממצא, ולא כישלון של הניתוח.
4. *קבוצות מחוזיות זעירות.* עמודות המחוזות מסומנות ב-n, משום שבירושלים יש n = 3 תחנות ובדרום n = 7;
   ממוצעים אלו אינם גדלים יציבים.

## אתחול סביבת העבודה

תא זה הופך את המחברת לניתנת להרצה הן משכפול מקומי של המאגר והן מסשן חדש של Google Colab: הוא מתקין
רק את החבילות החסרות בפועל, מאתר את שורש המאגר (או משכפל אותו ב-Colab), ויוצר את תיקיית `outputs/nb`
המשותפת שאליה כותבת כל מחברת בסדרה זו. ניתן להריצו שוב בבטחה.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## ספריות, קבועים ותיקיית הפלט של השלב

כל מה שמתואר להלן פועל על טבלה בת 67 שורות, ולכן אין במחברת זו שום שלב יקר חישובית - ההרצה המלאה
נמשכת פחות משנייה ואינה מצריכה דגימה או קירוב. הקבועים רוכזו כאן כדי שניתן יהיה לכוונן מחדש את
הניתוח במקום אחד:

- `TOP_QUINTILE = 0.80` - סף האחוזון שממנו תחנה נחשבת "גבוהה" במדד מסוים; הוא מניע הן את כללי
  הארכיטיפים והן את הספירה `top_quintile_centrality_count`.
- `PROFILE_TOP_N = 16` - מספר התחנות המופיעות במפת החום הרב-מדדית.

`CENTRALITY_FAMILIES` הוא הקיבוץ שבו אנו משתמשים לתיקון היתירות של ציון ה-composite: `degree`,
`weighted_degree` ו-`pagerank` מודדים כולם, בקירוב, "כמה שירות עובר כאן", ולכן הם ממוצעים *בתוך*
משפחה לפני שהמשפחות ממוצעות ביניהן.

בנוסף אנו יוצרים תיקיית שלב ייעודית למחברת זו. דבר אינו נכתב לעולם אל `outputs/tables`,
`outputs/figures` או `outputs/rail` - אלה מכילים את התוצאות המצוטטות בדוח הכתוב.

In [ ]:
_ensure("pandas", "numpy", "matplotlib", "seaborn", "scipy")

from itertools import cycle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

sns.set_theme(style="whitegrid", font_scale=1.0)  # must run BEFORE the Hebrew font patch

STAGE = OUT / "12_rail_socioeconomic"
(STAGE / "tables").mkdir(parents=True, exist_ok=True)
(STAGE / "figures").mkdir(parents=True, exist_ok=True)

# ---- tunable constants -------------------------------------------------------
TOP_QUINTILE = 0.80      # "high" on a metric = top 20% of the 67 rail stations
PROFILE_TOP_N = 16       # rows in the multi-metric profile heatmap
FIG_DPI = 200

CENTRALITY_METRICS = [
    "degree",
    "weighted_degree",
    "pagerank",
    "betweenness",
    "harmonic",
]
DAMAGE_METRIC = "stations_outside_largest_component"
PROFILE_METRICS = CENTRALITY_METRICS + [DAMAGE_METRIC]
PROFILE_LABELS = {
    "degree": "Degree",
    "weighted_degree": "Service volume",
    "pagerank": "PageRank",
    "betweenness": "Betweenness",
    "harmonic": "Harmonic",
    DAMAGE_METRIC: "Single-outage damage",
}

# Grouping used by the family-balanced composite score (fix #2).
CENTRALITY_FAMILIES = {
    "Service volume": ["degree", "weighted_degree", "pagerank"],
    "Brokerage": ["betweenness"],
    "Global accessibility": ["harmonic"],
}

print("Stage folder:", STAGE)

## תוויות בעברית ב-matplotlib

כל שם תחנה, שם מחוז ושם יישוב בפיד ה-GTFS הוא בעברית. matplotlib משרטט תווים בסדר לוגי (סדר האחסון)
ואינו מיישם את האלגוריתם הדו-כיווני של Unicode, ולכן העברית מוצגת הפוכה. התיקון שלהלן עוטף את
`Text.set_text` פעם אחת, כך *שכל* טקסט שנעביר ל-matplotlib - תוויות צירים, הערות וכותרות - מומר
אוטומטית לסדר התצוגה.

מכיוון שהתיקון הוא גלובלי, המשך המחברת מעביר ל-matplotlib מחרוזות עברית **גולמיות** ואינו קורא
ל-`fix_he` באופן ידני; קריאה ידנית נוספת הייתה הופכת את הטקסט פעמיים.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## איתור הקלטים

מחברת זו היא שלב *צרכן*: היא אינה מחשבת גרף משלה. היא זקוקה לשני דברים.

**1. תוצאות הרכבות ממחברת 11.** תחילה אנו מחפשים ב-`outputs/nb/11_*/tables/`. אם שלב זה לא הורץ
בסשן הנוכחי, אנו נסוגים לטבלאות הדוח המקובעות במאגר תחת `outputs/rail/tables` (לקריאה בלבד - איננו
כותבים לשם לעולם), ומודיעים על כך במפורש. אם אף אחד מהם אינו קיים, אנו מעלים שגיאה ברורה המנחה
להריץ את מחברת 11.

**2. ה-join הסוציו-אקונומי.** `stops_with_socioeconomic.csv` ממפה כל תחנת GTFS לאזור סטטיסטי של
הלמ"ס 2021 (אשכול 1-10, ערך המדד, יישוב, אוכלוסייה) בתוספת תווית מחוז מנהלי ומטרופולין. הפקתו
מחייבת את `geopandas` והורדה חיה של שכבת ה-ArcGIS של הלמ"ס, ולכן מדובר **בנכס נתונים מקובע במאגר**
ולא במשהו שמחושב כאן מחדש. תחילה אנו מחפשים אותו בכל תיקיית שלב של מחברת, ולאחר מכן נסוגים לעותק
המנוהל תחת מחברת 08, המורידה את שכבת הלמ"ס באופן חי וכותבת אותה לתיקיית השלב שלה. זוהי תלות בנתונים
ולא תלות בקוד: שום מודול מאותה תיקייה אינו מיובא.

In [ ]:
RAIL_TABLE_NAMES = ["rail_station_metrics.csv", "single_station_damage.csv"]


def _find_stage_dir(prefix):
    """Return the first outputs/nb folder whose name starts with `prefix`."""
    matches = sorted(p for p in OUT.glob(prefix + "*") if p.is_dir())
    return matches[0] if matches else None


def resolve_rail_tables():
    stage_11 = _find_stage_dir("11_")
    if stage_11 is not None:
        tables = stage_11 / "tables"
        if all((tables / name).exists() for name in RAIL_TABLE_NAMES):
            print("Rail metrics source: notebook 11 stage outputs ->", tables)
            return tables
    legacy = REPO / "outputs" / "rail" / "tables"
    if all((legacy / name).exists() for name in RAIL_TABLE_NAMES):
        print("Rail metrics source: committed report tables ->", legacy)
        print("  (notebook 11 stage output not found; these are read-only and never modified)")
        return legacy
    raise FileNotFoundError(
        f"Missing {RAIL_TABLE_NAMES} in {OUT}/11_*/tables and in {legacy} - "
        "run notebook 11 (rail network analysis) first."
    )


def resolve_socioeconomic_csv():
    """Locate the CBS stop-to-statistical-area join produced by notebook 08."""
    for candidate in sorted(OUT.glob("*/tables/stops_with_socioeconomic.csv")):
        print("Socioeconomic source: notebook stage output ->", candidate)
        return candidate
    raise FileNotFoundError(
        f"stops_with_socioeconomic.csv not found under {OUT}/*/tables/. "
        "It is produced by notebook 08 (socioeconomic equity), which downloads the "
        "CBS 2021 statistical-area layer and joins every GTFS stop to it. "
        "Run 08_socioeconomic_equity.ipynb first."
    )


RAIL_TABLES = resolve_rail_tables()
SOCIO_CSV = resolve_socioeconomic_csv()

metrics = pd.read_csv(
    RAIL_TABLES / "rail_station_metrics.csv", dtype={"stop_id": str}, encoding="utf-8-sig"
)
damage = pd.read_csv(
    RAIL_TABLES / "single_station_damage.csv", dtype={"stop_id": str}, encoding="utf-8-sig"
)[["stop_id", DAMAGE_METRIC, "components"]]
socioeconomic = pd.read_csv(
    SOCIO_CSV, dtype={"stop_id": str}, encoding="utf-8-sig", low_memory=False
)

print(f"\nrail stations: {len(metrics)} | damage rows: {len(damage)} | "
      f"stops in socioeconomic join: {len(socioeconomic):,}")
metrics.head(3)

## בניית טבלת הניתוח

אנו ממזגים את שלושת המקורות לפי `stop_id` וגוזרים את העמודות שבהן משתמשת שאר המחברת.

- **`socio_group`** מחלק את אשכול הלמ"ס לשלוש רצועות קריאות (הטווח הנצפה הוא 2-9, ולכן אין רצועות
  ריקות): נמוך = 2-4, בינוני = 5-7, גבוה = 8-9.
- **אחוזוני מרכזיות** מדרגים כל אחד מחמשת המדדים על פני *כל* התחנות באמצעות `method="average"`, כך
  שתיקו (נפוץ מאוד עבור `degree`) מקבל את הדירוג האמצעי המשותף.
- **אחוזוני נזק - התיקון.** הקוד המקורי הפיק עמודה אחת שהייתה אפס עבור תחנות שאינן מפרקות את הגרף,
  ואחוזון *בקרב המיעוט המפרק* עבור השאר. זוהי בחירת תצוגה סבירה, אך היא **אינה** אחוזון מבין 67,
  שזה בדיוק מה שסרגל הצבעים של מפת החום הישנה הצהיר. כעת אנו מחשבים את שניהם, תחת שמות שמצהירים על
  מהותם:
  - `damage_percentile_among_67` - מדורג על פני כל התחנות, וניתן להשוואה ישירה לחמשת אחוזוני
    המרכזיות (כל תחנה שאינה מפרקת נוחתת על אותו דירוג אמצעי משותף);
  - `damage_percentile_among_damaging` - ההגדרה המקורית, המדורגת רק בקרב התחנות שהסרתן אכן מנתקת
    משהו, כאשר האפסים מאולצים ל-0.
  מפת החום שבהמשך נמנעת מהעמימות לחלוטין באמצעות שרטוט הנזק ביחידות גולמיות על סקאלה נפרדת משלו.

אם תחנת רכבת כלשהי אינה מקבלת שיוך סוציו-אקונומי, אנו עוצרים בשגיאה קשיחה במקום לנתח בשקט תת-קבוצה
מוטה.

In [ ]:
SOCIO_COLUMNS = [
    "stop_id",
    "socio_locality",
    "socio_cluster",
    "socio_index_value",
    "socio_population",
    "socio_join_method",
    "socio_join_distance_m",
    "region",
    "metro",
]


def build_analysis_frame(metrics, damage, socioeconomic):
    frame = metrics.merge(damage, on="stop_id", how="left").merge(
        socioeconomic[SOCIO_COLUMNS], on="stop_id", how="left"
    )
    if frame["socio_cluster"].isna().any():
        missing = frame.loc[frame["socio_cluster"].isna(), "stop_name"].tolist()
        raise ValueError(f"Rail stations missing socioeconomic assignments: {missing}")

    # Booleans can survive a CSV round-trip as strings; normalise defensively.
    frame["is_articulation_point"] = (
        frame["is_articulation_point"].astype(str).str.strip().str.lower().isin(["true", "1"])
    )
    frame[DAMAGE_METRIC] = frame[DAMAGE_METRIC].fillna(0).astype(int)
    frame["components"] = frame["components"].fillna(1).astype(int)

    frame["socio_group"] = pd.cut(
        frame["socio_cluster"],
        bins=[0, 4, 7, 10],
        labels=["Lower (clusters 2-4)", "Middle (clusters 5-7)", "Higher (clusters 8-9)"],
    )

    for metric in CENTRALITY_METRICS:
        frame[f"{metric}_percentile"] = frame[metric].rank(method="average", pct=True)

    # (fix 1) Two explicitly named damage scales instead of one mislabelled one.
    frame["damage_percentile_among_67"] = frame[DAMAGE_METRIC].rank(
        method="average", pct=True
    )
    frame["damage_percentile_among_damaging"] = 0.0
    damaging = frame[DAMAGE_METRIC] > 0
    frame.loc[damaging, "damage_percentile_among_damaging"] = frame.loc[
        damaging, DAMAGE_METRIC
    ].rank(method="average", pct=True)

    frame["top_quintile_centrality_count"] = sum(
        frame[f"{metric}_percentile"].ge(TOP_QUINTILE).astype(int)
        for metric in CENTRALITY_METRICS
    )
    return frame


frame = build_analysis_frame(metrics, damage, socioeconomic)

n_damaging = int((frame[DAMAGE_METRIC] > 0).sum())
print(f"stations: {len(frame)}")
print(f"socioeconomic join: {(frame.socio_join_method == 'within').sum()} within-polygon, "
      f"{(frame.socio_join_method == 'nearest').sum()} nearest-polygon")
print(f"CBS cluster range: {frame.socio_cluster.min():.0f}-{frame.socio_cluster.max():.0f}")
print(f"stations whose single removal separates others: {n_damaging} of {len(frame)}")
print("\nWhy the two damage scales differ (a non-fragmenting station):")
cols = ["stop_name", DAMAGE_METRIC, "damage_percentile_among_67", "damage_percentile_among_damaging"]
print(frame.sort_values(DAMAGE_METRIC, ascending=False)[cols].head(3).to_string(index=False))
print(frame.loc[frame[DAMAGE_METRIC] == 0, cols].head(2).to_string(index=False))
frame["socio_group"].value_counts().sort_index()

## עד כמה חמשת מדדי המרכזיות יתירים? (תיקון 2)

לפני שאנו משלבים מדדים למספר יחיד, ראוי לבדוק אם הם מודדים דברים שונים. מטריצת Spearman שלהלן מראה
שברובם אינם: `weighted_degree` ו-`scheduled_stop_calls` מתואמים ב-rho ~ 0.94 (הם כמעט אותו גודל -
הדרגה המשוקללת של תחנה בגרף שירות *היא* עומס התנועה שלה), ו-`pagerank` עומד על ~0.86 מול
`weighted_degree`. לפיכך, ה-`composite_centrality_score` הישן, ממוצע פשוט של חמישה אחוזונים, העניק
בפועל לנפח השירות שלושה קולות מתוך חמישה, בעוד שתיווך (brokerage) ונגישות קיבלו קול אחד כל אחד.

לכן אנו מחשבים שני ציונים ושומרים את שניהם בטבלה המיוצאת:

- `composite_centrality_score_legacy` - הממוצע הפשוט המקורי של חמישה אחוזונים (נשמר כדי שאיורי הדוח
  הקודמים יישארו ניתנים לשחזור);
- `composite_centrality_score_balanced` - ממוצע *בתוך* כל משפחה ב-`CENTRALITY_FAMILIES` תחילה, ולאחר
  מכן ממוצע של שלוש המשפחות, כך שנפח / תיווך / נגישות מקבלים משקל של 1:1:1. זהו הציון שלפיו מדורגת
  שאר המחברת.

התא מדפיס עד כמה שני הדירוגים אכן נבדלים זה מזה, כך שהבחירה ניתנת לביקורת ואינה מוצהרת בלבד.

In [ ]:
redundancy = frame[PROFILE_METRICS + ["scheduled_stop_calls"]].corr(method="spearman")
print("Spearman redundancy matrix (rail metrics):")
print(redundancy.round(3).to_string())
print(
    "\nweighted_degree vs scheduled_stop_calls: rho = "
    f"{redundancy.loc['weighted_degree', 'scheduled_stop_calls']:.3f}"
)

percentile_columns = [f"{m}_percentile" for m in CENTRALITY_METRICS]
frame["composite_centrality_score_legacy"] = 100 * frame[percentile_columns].mean(axis=1)

family_scores = pd.DataFrame(
    {
        family: frame[[f"{m}_percentile" for m in members]].mean(axis=1)
        for family, members in CENTRALITY_FAMILIES.items()
    },
    index=frame.index,
)
frame["composite_centrality_score_balanced"] = 100 * family_scores.mean(axis=1)
# Canonical alias used by the ranking below.
frame["composite_centrality_score"] = frame["composite_centrality_score_balanced"]

composite_comparison = frame[["stop_id", "stop_name"]].copy()
composite_comparison["score_legacy"] = frame["composite_centrality_score_legacy"]
composite_comparison["score_balanced"] = frame["composite_centrality_score_balanced"]
composite_comparison["rank_legacy"] = (
    frame["composite_centrality_score_legacy"].rank(ascending=False, method="min").astype(int)
)
composite_comparison["rank_balanced"] = (
    frame["composite_centrality_score_balanced"].rank(ascending=False, method="min").astype(int)
)
composite_comparison["rank_shift"] = (
    composite_comparison["rank_legacy"] - composite_comparison["rank_balanced"]
)

rho_scores = spearmanr(
    composite_comparison["score_legacy"], composite_comparison["score_balanced"]
).statistic
print(f"\nLegacy vs family-balanced composite: Spearman rho = {rho_scores:.3f}")
print("Largest ranking movers (positive shift = the balanced score ranks the station higher):")
movers = composite_comparison.reindex(
    composite_comparison["rank_shift"].abs().sort_values(ascending=False).index
).head(8)
print(movers[["stop_name", "rank_legacy", "rank_balanced", "rank_shift"]].to_string(index=False))

## ארכיטיפים של תחנות

ציון מדורג יחיד מסתיר את העובדה שתחנות כושלות ב*אופנים* שונים. הכלל שלהלן (הועבר ללא שינוי מהניתוח
המקורי) מתייג כל תחנה על סמך שלושה אותות בינאריים:

- **שירות גבוה** - חמישון עליון הן ב-`weighted_degree` והן ב-`pagerank`;
- **תיווך גבוה** - חמישון עליון ב-`betweenness`;
- **חתך מזיק** - התחנה היא נקודת חיתוך (articulation point) *וגם* הסרתה מנתקת בפועל לפחות תחנה אחת
  נוספת מהרכיב הגדול ביותר.

התאים המעניינים הם הצירופים: *מרכזת שירות גבוה קריטית* היא עמוסה **וגם** צומת חיתוך (המקרה הגרוע
ביותר), *מרכזת שירות גבוה יתירה* היא עמוסה אך הרשת מנתבת סביבה, ו*שער ענף* הוא תחנה שקטה שאף על פי
כן מחזיקה ענף שלם מחובר לרשת - בדיוק סוג הפגיעות שדירוג המבוסס על תנועה בלבד היה מפספס.

In [ ]:
def classify_station(row):
    high_service = (
        row["weighted_degree_percentile"] >= TOP_QUINTILE
        and row["pagerank_percentile"] >= TOP_QUINTILE
    )
    high_brokerage = row["betweenness_percentile"] >= TOP_QUINTILE
    damaging_cut = bool(row["is_articulation_point"]) and row[DAMAGE_METRIC] > 0
    if high_service and damaging_cut:
        return "critical high-service hub"
    if high_service and not damaging_cut:
        return "redundant high-service hub"
    if high_brokerage and damaging_cut:
        return "structural bottleneck"
    if high_brokerage:
        return "structural broker with alternatives"
    if damaging_cut:
        return "branch gateway"
    if row["harmonic_percentile"] >= TOP_QUINTILE:
        return "globally accessible connector"
    return "ordinary or peripheral station"


frame["station_archetype"] = frame.apply(classify_station, axis=1)
print(frame["station_archetype"].value_counts().to_string())
print()
print("Archetype by socioeconomic group (station counts):")
print(
    pd.crosstab(frame["station_archetype"], frame["socio_group"]).to_string()
)

## איור 1 - פרופילים רב-מדדיים של התחנות המובילות (תיקון 1 מיושם)

אנו לוקחים את `PROFILE_TOP_N` התחנות בעלות מספר ההופעות הגדול ביותר בחמישון העליון (שוויונות מוכרעים
לפי ה-composite המאוזן-משפחות) ומציגים את האחוזון שלהן בכל מדד מרכזיות.

עמודת הנזק מתקלה בודדת **אינה** מוצגת כאחוזון באותה סקאלת צבעים. האיור הישן עשה זאת, וערבב דירוג על
פני 67 תחנות עם דירוג על פני ~14 התחנות המפרקות, תחת סרגל צבעים שהצהיר "אחוזון מבין 67". כאן הפאנל
השמאלי מציג אחוזונים על פני כל 67 התחנות (סרגל צבעים אחד, משמעות אחת), והפאנל הימני מציג את הספירה
הגולמית של התחנות המנותקות עקב הסרת אותה תחנה, עם סרגל צבעים משלו ביחידות של תחנות. קריאה לרוחב שורה
משווה אפוא בין דומים בצד שמאל, ומספקת גודל מוחלט בצד ימין.

In [ ]:
def plot_centrality_profiles(frame, path):
    ranking = frame.sort_values(
        ["top_quintile_centrality_count", "composite_centrality_score_balanced"],
        ascending=False,
    ).head(PROFILE_TOP_N)

    percentiles = ranking[[f"{m}_percentile" for m in CENTRALITY_METRICS]].copy()
    percentiles.columns = [PROFILE_LABELS[m] for m in CENTRALITY_METRICS]
    percentiles.index = list(ranking["stop_name"])

    damage_panel = ranking[[DAMAGE_METRIC]].copy()
    damage_panel.columns = [PROFILE_LABELS[DAMAGE_METRIC]]
    damage_panel.index = list(ranking["stop_name"])

    fig, axes = plt.subplots(
        1, 2, figsize=(13, 8), gridspec_kw={"width_ratios": [5, 1.35]}
    )
    sns.heatmap(
        percentiles,
        annot=True,
        fmt=".0%",
        cmap="viridis",
        vmin=0,
        vmax=1,
        linewidths=0.4,
        cbar_kws={"label": "Percentile among all 67 rail stations"},
        ax=axes[0],
    )
    axes[0].set_title("Centrality percentiles (ranked over all 67 stations)")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("")

    sns.heatmap(
        damage_panel,
        annot=True,
        fmt=".0f",
        cmap="rocket_r",
        vmin=0,
        vmax=int(frame[DAMAGE_METRIC].max()),
        linewidths=0.4,
        yticklabels=False,
        cbar_kws={"label": "Stations separated (count, out of 66)"},
        ax=axes[1],
    )
    axes[1].set_title("Single-outage damage\n(raw units, different scale)", fontsize=10)
    axes[1].set_xlabel("")
    axes[1].set_ylabel("")

    fig.suptitle("Multi-metric profiles of Israel Railways' leading stations")
    fig.tight_layout()
    fig.savefig(path, dpi=FIG_DPI)
    return fig


fig = plot_centrality_profiles(frame, STAGE / "figures" / "centrality_profile_heatmap.png")
plt.show()

## איור 2 - עומס תפעולי מול פגיעות מבנית

תרשים פיזור זה עונה ישירות על שאלה תכנונית: האם התחנות שכשלן היה מזיק ביותר הן אותן תחנות שהן
העמוסות ביותר? ציר ה-x הוא מספר עצירות מתוזמנות (סקאלה לוגריתמית, משום שהתנועה משתרעת על פני סדרי
גודל), ציר ה-y הוא מספר התחנות האחרות שתקלה בודדת ממודלת מנתקת מהרכיב הגדול ביותר, גודל הבועה הוא
betweenness והצבע הוא האשכול הסוציו-אקונומי המקומי של הלמ"ס.

נקודות לאורך התחתית הן תחנות עמוסות אך יתירות; נקודות גבוהות בצד שמאל הן שערי הענף השקטים. אנו
מתייגים כל תחנה שמנתקת 3 תחנות או יותר, או שנמצאת ב-8% העליונים של התנועה.
(תיקון עמידות קטן ביחס למקור: היסטי התוויות מסובבים מחזורית במקום להיות מוצמדים לרשימה קבועה של 8,
דבר שהשמיט תוויות בשקט כאשר יותר משמונה תחנות עמדו בתנאי.)

In [ ]:
def plot_service_vs_damage(frame, path):
    fig, axis = plt.subplots(figsize=(10, 7))
    sizes = 45 + 650 * frame["betweenness"] / max(frame["betweenness"].max(), 1e-12)
    scatter = axis.scatter(
        frame["scheduled_stop_calls"],
        frame[DAMAGE_METRIC],
        c=frame["socio_cluster"],
        s=sizes,
        cmap="viridis",
        vmin=1,
        vmax=10,
        alpha=0.82,
        edgecolor="white",
        linewidth=0.7,
    )
    labels = frame[
        (frame[DAMAGE_METRIC] >= 3)
        | (frame["scheduled_stop_calls"] >= frame["scheduled_stop_calls"].quantile(0.92))
    ].sort_values(DAMAGE_METRIC, ascending=False)
    offsets = cycle(
        [(-8, 8), (-8, -12), (-6, 9), (6, -12), (-6, 9), (6, 8), (-6, -12), (6, 8)]
    )
    for (_, row), offset in zip(labels.iterrows(), offsets):
        axis.annotate(
            row["stop_name"],
            (row["scheduled_stop_calls"], row[DAMAGE_METRIC]),
            xytext=offset,
            textcoords="offset points",
            fontsize=8,
            ha="left" if offset[0] > 0 else "right",
            arrowprops={"arrowstyle": "-", "lw": 0.5, "color": "#64748b"},
        )
    axis.axvline(frame["scheduled_stop_calls"].median(), color="#94a3b8", ls="--", lw=1)
    axis.set_xscale("log")
    axis.set_xlabel("Scheduled stop calls (log scale)")
    axis.set_ylabel("Other stations separated by one modeled outage")
    axis.set_title("Operational busyness versus structural rail vulnerability")
    axis.margins(x=0.09, y=0.12)
    axis.grid(alpha=0.2)
    colorbar = fig.colorbar(scatter, ax=axis)
    colorbar.set_label("Local CBS socioeconomic cluster")
    fig.text(
        0.5,
        0.01,
        "Bubble size = betweenness centrality; colour = CBS cluster of the assigned statistical area.",
        ha="center",
        fontsize=9,
    )
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(path, dpi=FIG_DPI)
    return fig


fig = plot_service_vs_damage(frame, STAGE / "figures" / "service_vs_structural_damage.png")
plt.show()

## קבוצות סוציו-אקונומיות: טבלת סיכום ופרופיל מרכזיות

כעת נעסוק בשאלת השוויוניות ישירות. אנו מצרפים את התחנות לשלוש הרצועות הסוציו-אקונומיות ומשווים שירות
ומבנה, ולאחר מכן משרטטים את האחוזון הממוצע של כל מדד מרכזיות לכל רצועה.

יש לקרוא את תרשים הפרופיל ביחס לקו המקווקו של 50%: אילו המרכזיות ברשת הרכבות הייתה ניטרלית מבחינה
סוציו-אקונומית, שלוש העקומות היו מרחפות סביבו. שימו לב לגדלי הקבוצות (18 / 28 / 21 תחנות) ולעמודה
`within_polygon_share` - כשליש מהתחנות הותאמו לאזור הסטטיסטי *הקרוב ביותר* ולא נפלו בתוך אזור כזה,
משום שתחנות רכבת נבנות לעיתים קרובות בשולי השטח הבנוי. זהו מקור רעש אמיתי בתווית הסוציו-אקונומית,
ולכן סעיף המתאמים חוזר על כל מבחן גם על תת-הקבוצה שבתוך הפוליגון בלבד.

In [ ]:
def group_summary(frame):
    summary = (
        frame.groupby("socio_group", observed=True)
        .agg(
            stations=("stop_id", "size"),
            within_polygon_share=("socio_join_method", lambda v: (v == "within").mean()),
            median_socio_cluster=("socio_cluster", "median"),
            mean_degree=("degree", "mean"),
            median_weighted_degree=("weighted_degree", "median"),
            mean_scheduled_stop_calls=("scheduled_stop_calls", "mean"),
            median_scheduled_stop_calls=("scheduled_stop_calls", "median"),
            total_scheduled_stop_calls=("scheduled_stop_calls", "sum"),
            mean_pagerank=("pagerank", "mean"),
            mean_betweenness=("betweenness", "mean"),
            median_betweenness=("betweenness", "median"),
            mean_harmonic=("harmonic", "mean"),
            articulation_point_share=("is_articulation_point", "mean"),
            stations_causing_fragmentation=(DAMAGE_METRIC, lambda v: (v > 0).sum()),
            mean_single_station_damage=(DAMAGE_METRIC, "mean"),
        )
        .reset_index()
    )
    summary["scheduled_stop_call_share"] = (
        summary["total_scheduled_stop_calls"] / summary["total_scheduled_stop_calls"].sum()
    )
    return summary


def plot_socioeconomic_profiles(frame, path):
    columns = [f"{m}_percentile" for m in CENTRALITY_METRICS]
    profile = frame.groupby("socio_group", observed=True)[columns].mean().T
    profile.index = [PROFILE_LABELS[m] for m in CENTRALITY_METRICS]
    counts = frame["socio_group"].value_counts()

    fig, axis = plt.subplots(figsize=(9, 6))
    for marker, group in zip(["o", "s", "^"], profile.columns):
        axis.plot(
            profile.index,
            100 * profile[group],
            marker=marker,
            linewidth=2,
            markersize=7,
            label=f"{group}  (n={int(counts[group])})",
        )
    axis.axhline(50, color="#94a3b8", ls="--", lw=1)
    axis.set_ylabel("Mean station percentile")
    axis.set_title("Rail centrality profiles by local socioeconomic group")
    low = float((100 * profile.values).min())
    high = float((100 * profile.values).max())
    axis.set_ylim(min(low, 50) - 8, max(high, 50) + 8)
    axis.grid(axis="y", alpha=0.2)
    axis.legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=FIG_DPI)
    return fig


socioeconomic_groups = group_summary(frame)
display_cols = [
    "socio_group", "stations", "within_polygon_share", "mean_degree",
    "mean_scheduled_stop_calls", "mean_betweenness", "articulation_point_share",
    "stations_causing_fragmentation", "scheduled_stop_call_share",
]
print(socioeconomic_groups[display_cols].round(3).to_string(index=False))

fig = plot_socioeconomic_profiles(
    frame, STAGE / "figures" / "socioeconomic_centrality_profiles.png"
)
plt.show()

## השוואה מחוזית - עם גדלי מדגם על התרשים (תיקון 4)

אותה השוואה על פני ארבעת המחוזות המנהליים שבהם נעשה שימוש לאורך כל הפרויקט (מרכז / צפון / דרום /
ירושלים). שני פאנלים: ממוצע עצירות מתוזמנות לתחנה (עצימות שירות) ושיעור תחנות המחוז שהן נקודות חיתוך
(שבריריות מבנית).

**גדלי המדגם חשובים יותר מהעמודות עצמן.** במרכז יש 39 תחנות, בצפון 18, בדרום 7 ובירושלים 3. "שיעור
נקודות חיתוך" המחושב על פני 3 תחנות זז בקפיצות של 33 נקודות אחוז - תחנה בודדת משנה את העמודה
לחלוטין. לפיכך אנו מסמנים את n על כל עמודה ומסמנים את הקבוצות הקטנות, במקום להזמין את הקורא להשוות
גבהים כאילו היו מדויקים באותה מידה.

In [ ]:
def regional_summary(frame):
    summary = (
        frame.groupby("region", dropna=False)
        .agg(
            stations=("stop_id", "size"),
            mean_socio_cluster=("socio_cluster", "mean"),
            mean_scheduled_stop_calls=("scheduled_stop_calls", "mean"),
            total_scheduled_stop_calls=("scheduled_stop_calls", "sum"),
            mean_betweenness=("betweenness", "mean"),
            mean_harmonic=("harmonic", "mean"),
            articulation_point_share=("is_articulation_point", "mean"),
            mean_single_station_damage=(DAMAGE_METRIC, "mean"),
        )
        .reset_index()
    )
    return summary.sort_values("mean_betweenness", ascending=False)


def plot_regional_profile(summary, path, small_group_threshold=10):
    ordered = summary.sort_values("mean_scheduled_stop_calls", ascending=False)
    labels = list(ordered["region"])
    counts = list(ordered["stations"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    panels = [
        (ordered["mean_scheduled_stop_calls"], "#2563eb",
         "Mean scheduled calls per station", "Scheduled stop calls"),
        (100 * ordered["articulation_point_share"], "#dc2626",
         "Stations that are articulation points", "Share of regional stations (%)"),
    ]
    for axis, (values, colour, title, ylabel) in zip(axes, panels):
        bars = axis.bar(labels, values, color=colour)
        axis.set_title(title)
        axis.set_ylabel(ylabel)
        axis.tick_params(axis="x", rotation=20)
        axis.grid(axis="y", alpha=0.2)
        axis.margins(y=0.18)
        # (fix 4) n on every bar; small groups flagged explicitly.
        for bar, n in zip(bars, counts):
            note = f"n={n}" + (" *" if n < small_group_threshold else "")
            axis.annotate(
                note,
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center",
                fontsize=9,
                color="#334155",
            )
    fig.suptitle("Regional rail service intensity and structural fragility")
    fig.text(
        0.5,
        0.01,
        f"* fewer than {small_group_threshold} stations - these regional means are unstable "
        "and should not be compared as point estimates.",
        ha="center",
        fontsize=9,
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(path, dpi=FIG_DPI)
    return fig


regions = regional_summary(frame)
print(regions.round(3).to_string(index=False))

fig = plot_regional_profile(regions, STAGE / "figures" / "regional_rail_profile.png")
plt.show()

## מבחן פורמלי: האם מרכזיות קשורה למעמד סוציו-אקונומי? (תיקון 3)

אנו מריצים מתאמי דירוג Spearman בין שני משתנים סוציו-אקונומיים (`socio_cluster`, רצועת הלמ"ס 1-10,
ו-`socio_index_value`, המדד הרציף) לבין שישה מדדי רשת, על שני מדגמים (כל 67 התחנות, ו-44 התחנות
שהותאמו באופן מובהק בתוך פוליגון של אזור סטטיסטי). כלומר 2 x 6 x 2 = **24 מבחני השערה על אותן 67
תחנות**.

הרצת 24 מבחנים ב-alpha = 0.05 מעניקה סיכוי של כ-70% לקבל לפחות תוצאה "מובהקת" אחת תחת השערת אפס
אמיתית, ולכן ערך p גולמי כאן משמעותו מועטה מאוד. אנו מוסיפים שני תיקונים סטנדרטיים:

- **Benjamini-Hochberg** (`q_value_bh`) השולט בשיעור התגליות השגויות (false discovery rate);
- **Holm-Bonferroni** (`p_value_holm`) השולט בשיעור השגיאה המשפחתי (family-wise error rate).

שניהם ממומשים ישירות בתוך התא בכמה שורות, כך שהמחברת נשארת דלת-תלויות. המבחנים גם אינם בלתי תלויים -
ששת המדדים מתואמים ביניהם בחוזקה (ראו את מטריצת היתירות) ושני המדגמים חופפים - ולכן התיקונים
שמרניים בכיוון אחד, ומספר המבחנים הבלתי תלויים האפקטיבי נמוך בהרבה מ-24. דבר מכל זה אינו משנה את
המסקנה, כפי שמראה הפלט.

In [ ]:
def benjamini_hochberg(p_values):
    """BH false-discovery-rate adjusted p-values (q-values)."""
    p = np.asarray(p_values, dtype=float)
    n = p.size
    order = np.argsort(p)
    scaled = p[order] * n / np.arange(1, n + 1)
    monotone = np.minimum.accumulate(scaled[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(monotone, 0, 1)
    return out


def holm_bonferroni(p_values):
    """Holm-Bonferroni family-wise-error adjusted p-values."""
    p = np.asarray(p_values, dtype=float)
    n = p.size
    order = np.argsort(p)
    scaled = p[order] * (n - np.arange(n))
    monotone = np.maximum.accumulate(scaled)
    out = np.empty(n)
    out[order] = np.clip(monotone, 0, 1)
    return out


def correlation_table(frame):
    rows = []
    samples = {
        "all assignments": frame,
        "within-polygon only": frame[frame["socio_join_method"] == "within"],
    }
    for sample_name, sample in samples.items():
        for socio_variable in ["socio_cluster", "socio_index_value"]:
            for metric in PROFILE_METRICS:
                subset = sample[[socio_variable, metric]].dropna()
                rho, p_value = spearmanr(subset[socio_variable], subset[metric])
                rows.append(
                    {
                        "sample": sample_name,
                        "socioeconomic_variable": socio_variable,
                        "network_metric": metric,
                        "spearman_rho": float(rho),
                        "p_value_two_sided": float(p_value),
                        "n_stations": len(subset),
                    }
                )
    table = pd.DataFrame(rows)
    table["q_value_bh"] = benjamini_hochberg(table["p_value_two_sided"])
    table["p_value_holm"] = holm_bonferroni(table["p_value_two_sided"])
    table["significant_after_correction"] = table["q_value_bh"] < 0.05
    return table


correlations = correlation_table(frame)
print(
    correlations.round(4).to_string(index=False)
)

strongest = correlations.loc[correlations["spearman_rho"].abs().idxmax()]
smallest_p = correlations.loc[correlations["p_value_two_sided"].idxmin()]
print("\n--- honest summary of the 24 tests ---")
print(f"tests run: {len(correlations)}")
print(f"raw p < 0.05: {(correlations['p_value_two_sided'] < 0.05).sum()}")
print(f"BH q < 0.05: {(correlations['q_value_bh'] < 0.05).sum()}")
print(f"Holm p < 0.05: {(correlations['p_value_holm'] < 0.05).sum()}")
print(
    f"largest |rho|: {strongest.spearman_rho:+.3f} "
    f"({strongest.network_metric}, {strongest.socioeconomic_variable}, {strongest['sample']}, "
    f"raw p={strongest.p_value_two_sided:.3f})"
)
print(
    f"smallest raw p: {smallest_p.p_value_two_sided:.3f} "
    f"({smallest_p.network_metric}, {smallest_p['sample']}) -> "
    f"BH q={smallest_p.q_value_bh:.3f}"
)
print(
    "Interpretation: with n=67 (44 within-polygon) even a moderate true effect of |rho| ~ 0.24 "
    "would be needed for raw significance, so this is an underpowered null result, not evidence "
    "of exact equality."
)

## שמירת תוצרי השלב

כל מה שנכתב כאן נשמר בתיקיית השלב הייעודית של מחברת זו,
`outputs/nb/12_rail_socioeconomic/`. הטבלה ברמת התחנה ממוינת לפי ה-composite המאוזן-משפחות ושומרת את
*שני* ציוני ה-composite ואת *שתי* ההגדרות של אחוזון הנזק, כך שקורא יוכל לשחזר כל אחת מהמוסכמות מבלי
להריץ מחדש את המחברת. קובצי ה-CSV נכתבים בקידוד `utf-8-sig` כדי ששמות תחנות בעברית ייפתחו כראוי
ב-Excel.

In [ ]:
tables_dir = STAGE / "tables"

frame.sort_values("composite_centrality_score_balanced", ascending=False).to_csv(
    tables_dir / "rail_station_archetypes.csv", index=False, encoding="utf-8-sig"
)
correlations.to_csv(
    tables_dir / "rail_socioeconomic_correlations.csv", index=False, encoding="utf-8-sig"
)
redundancy.to_csv(tables_dir / "rail_centrality_spearman.csv", encoding="utf-8-sig")
composite_comparison.sort_values("rank_balanced").to_csv(
    tables_dir / "composite_score_comparison.csv", index=False, encoding="utf-8-sig"
)
socioeconomic_groups.to_csv(
    tables_dir / "rail_socioeconomic_group_summary.csv", index=False, encoding="utf-8-sig"
)
regions.to_csv(
    tables_dir / "rail_regional_summary.csv", index=False, encoding="utf-8-sig"
)

for path in sorted(STAGE.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUT), f"({path.stat().st_size / 1024:.1f} KB)")

## מסקנות

**1. הממצא המרכזי הוא ממצא שלילי, ואנו מדווחים עליו ככזה.** על פני 24 מבחני Spearman (שני משתנים
סוציו-אקונומיים x שישה מדדי רשת x שני מדגמי join) דבר אינו מגיע למובהקות אף *לפני* תיקון לריבוי
השוואות; הקשר החזק ביותר עומד על כ-|rho| ~ 0.27 וערך ה-p הגולמי הקטן ביותר הוא ~0.08, ההופך לערך
q של BH הגבוה בהרבה מ-0.05. במערך נתונים זה **אין קשר שיטתי בר-גילוי בין מרכזיותה של תחנת רכבת לבין
האשכול הסוציו-אקונומי של האזור הסטטיסטי שלה.** עם 67 תחנות (44 מהן הותאמו באופן מובהק בתוך פוליגון)
המחקר סובל מהספק סטטיסטי נמוך: רק אפקטים בסדר גודל של |rho| > 0.24 היו ניתנים לגילוי. לפיכך הניסוח
הנכון הוא "אין עדות לקשר בגודל מדגם זה", ולא "הוכחה לשוויוניות".

**2. האותות החלשים שכן קיימים מצביעים כולם לאותו כיוון, וראוי לציינם באופן תיאורי.** כל המתאמים בין
האשכול לבין degree, נגישות הרמונית ונזק מתקלה בודדת הם *שליליים*, כלומר אם כבר, תחנות באזורים בעלי
אשכול נמוך יותר ממוקמות בעמדות מקושרות מעט יותר ושבריריות מעט יותר - ההפך מהציפייה הנאיבית. המדד
היחיד החיובי באופן עקבי הוא weighted degree (נפח שירות), הנשלט על ידי ליבת תל אביב.

**3. נפח שירות אינו זהה לקריטיות מבנית.** איור 2 מפריד בין שני מצבי כשל נבדלים: מרכזות עמוסות שהרשת
מסוגלת לנתב סביב אובדנן ("מרכזת שירות גבוה יתירה") לעומת שערי ענף שקטים שאובדנם מנתק ענף שלם. רק 7
תחנות הן *מרכזות שירות גבוה קריטיות* (עמוסות **וגם** צומת חיתוך מזיקה); 6 נוספות הן שערי ענף בעלי
תנועה נמוכה שרשימת עדיפויות המבוססת על תנועה הייתה מפספסת לחלוטין. זהו הממצא בעל התועלת התפעולית של
מחברת זו.

**4. ציוני composite הם שבריריים מעצם בנייתם.** מכיוון ש-`weighted_degree`, `pagerank`
ו-`scheduled_stop_calls` הם כמעט כפילויות (rho עד 0.94), הממוצע הפשוט של חמישה מדדים משקלל בשקט את
נפח השירות ביחס של 3:1:1 מול תיווך ונגישות. הציון המאוזן-משפחות עדיין מתואם מאוד בדירוג עם הציון
המקורי, אך תחנות בודדות זזות מספר מקומות בדירוג. יש לקרוא כל "דירוג חשיבות" מסוג זה כסדר אחד בר-הגנה
מתוך כמה אפשריים, ולא כסדר היחיד.

**5. שתי הסתייגויות נתונים מגבילות את המרחק שאליו ניתן לקחת כל טענת שוויוניות.** כשליש מתחנות הרכבת
(23 מתוך 67) נאלצו להיות מותאמות לאזור הסטטיסטי *הקרוב ביותר* ולא לאזור המכיל אותן, משום שתחנות
נבנות לעיתים קרובות בשולי השטח הבנוי - ולכן התווית הסוציו-אקונומית שלהן רועשת. בנוסף, הפילוח המחוזי
נשען על קבוצות קטנות מאוד (ירושלים n = 3, דרום n = 7), ולכן עמודות אלו נושאות n מפורש ואזהרה במקום
להיות מושוות כאומדנים נקודתיים.